# Chapter 1 — Foundations: Code Companion

This notebook makes the ideas in `notes.md` concrete. Run cell by cell.

We'll use:
- `numpy` for vectors and arithmetic
- `matplotlib` for pictures in ℝ²

Run from the project root: `uv run jupyter lab`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Vectors as NumPy arrays

A vector in ℝⁿ is just a 1D NumPy array of length n.

In [ ]:
u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])

print('u      =', u)
print('v      =', v)
print('u + v  =', u + v)
print('2u - v =', 2*u - v)
print('shape  =', u.shape, '(so u lives in R^{})'.format(u.shape[0]))

## 2. The parallelogram rule, drawn

Worked Example 1.3: visualize `u`, `v`, and `u + v` as arrows from the origin.

In [ ]:
def draw_arrow(ax, vec, color, label, start=(0, 0)):
    ax.annotate('', xy=(start[0] + vec[0], start[1] + vec[1]),
                xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(start[0] + vec[0]*1.05, start[1] + vec[1]*1.05, label,
            color=color, fontsize=12, fontweight='bold')

fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.grid(True, alpha=0.3)

draw_arrow(ax, u,        'tab:blue',   'u')
draw_arrow(ax, v,        'tab:orange', 'v')
draw_arrow(ax, u + v,    'tab:green',  'u + v')
# Tip-to-tail picture: v starting at u's tip
draw_arrow(ax, v, 'tab:orange', '', start=u)
draw_arrow(ax, u, 'tab:blue',   '', start=v)

ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title('Vector addition: the parallelogram rule')
plt.show()

## 3. Scalar multiplication: stretch, shrink, flip

Multiplying by a positive scalar stretches; by a negative scalar flips.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.grid(True, alpha=0.3)

w = np.array([2.0, 1.0])
scalars = [-1.5, -0.5, 1.0, 1.5, 2.5]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(scalars)))

for c, color in zip(scalars, colors):
    draw_arrow(ax, c * w, color, f'{c} · w')

ax.set_xlim(-5, 6)
ax.set_ylim(-3, 4)
ax.set_aspect('equal')
ax.set_title('Scalar multiples of w = (2, 1)')
plt.show()

## 4. A linearity tester

Given a Python function `f`, randomly probe additivity and homogeneity to *empirically* check whether it could be linear. (This isn't a proof — but it's a useful sanity check.)

In [ ]:
def probably_linear(f, dim_in, dim_out=None, trials=20, tol=1e-9, seed=0):
    """Empirically test whether f: R^dim_in -> R^dim_out is linear.
    Returns True if all random probes pass; False otherwise."""
    rng = np.random.default_rng(seed)
    for _ in range(trials):
        x = rng.normal(size=dim_in)
        y = rng.normal(size=dim_in)
        c1, c2 = rng.normal(size=2)
        lhs = f(c1*x + c2*y)
        rhs = c1*np.asarray(f(x)) + c2*np.asarray(f(y))
        if not np.allclose(lhs, rhs, atol=tol):
            return False
    return True

# Test cases from Worked Example 1.4
tests = {
    'f(x,y) = 3x + 4y':       (lambda v: 3*v[0] + 4*v[1],            2),
    'f(x,y) = 3x + 4y + 7':   (lambda v: 3*v[0] + 4*v[1] + 7,        2),
    'f(x,y) = x*y':           (lambda v: v[0]*v[1],                  2),
    'f(x,y) = max(x,y)':      (lambda v: max(v[0], v[1]),            2),
    'f(x,y) = (x+y, x-y)':    (lambda v: np.array([v[0]+v[1], v[0]-v[1]]), 2),
}

for name, (f, n) in tests.items():
    print(f'{name:30s}  linear? {probably_linear(f, n)}')

## 5. "Linear ⇒ determined by basis vectors" — the bridge to matrices

From Worked Example 1.6: if `T : ℝ³ → ℝ²` is linear and we know `T(e₁), T(e₂), T(e₃)` (the standard basis vectors), we can compute T anywhere.

Stack those three output vectors as **columns** of a 2×3 matrix. Multiplying that matrix by any input vector gives `T(input)`. This is your first encounter with the matrix of a linear transformation — Chapter 4 will make it formal.

In [ ]:
# Define T directly
def T(v):
    x, y, z = v
    return np.array([x + 2*y, y - z])

# Standard basis vectors of R^3
e1 = np.array([1.0, 0.0, 0.0])
e2 = np.array([0.0, 1.0, 0.0])
e3 = np.array([0.0, 0.0, 1.0])

# Stack T(e1), T(e2), T(e3) as columns of a matrix
A = np.column_stack([T(e1), T(e2), T(e3)])
print('Matrix of T:')
print(A)
print()

# Now use the matrix to compute T on a random input
v = np.array([2.0, 3.0, -1.0])
print('T(v) directly       :', T(v))
print('A @ v (matrix form) :', A @ v)
print('Agree?              :', np.allclose(T(v), A @ v))

## 6. Optional — high-dimensional sanity

We can't *picture* ℝ¹⁰⁰, but the algebra works exactly as in ℝ². Build a vector in ℝ¹⁰⁰, scale it, add another, and confirm the result is what the formulas say.

In [ ]:
rng = np.random.default_rng(42)
n = 100
u100 = rng.normal(size=n)
v100 = rng.normal(size=n)

result = 2.5 * u100 - 3.0 * v100
print(f'A vector in R^{n} has shape {result.shape}.')
print(f'First five components: {result[:5]}')
print(f'No picture exists, but the arithmetic is identical to R^2.')

---

## What's next

Chapter 2 dives deeper into vector geometry — dot products, projections, angles. Chapter 3 starts solving systems `Ax = b` systematically. The matrix you saw in section 5 above is the bridge.

Read `notes.md`, work `worked-examples.md`, do `exercises.md`, then re-run this notebook with your own examples.